# 04 — Holt–Winters exponential smoothing (ETS)

**Data:** hourly `load_mw` from `data/raw/de_lu_load_hourly.parquet`.

1. **`statsmodels.tsa.holtwinters.ExponentialSmoothing`** with **seasonal period 24** on hourly series; **forecast 48 steps** ahead.
2. Same on **daily totals** (`resample('1D').sum()`), **seasonal period 7**; **forecast 7 days** ahead (one seasonal cycle).
3. **`seasonal='add'`** vs **`seasonal='mul'`** (trend `'add'`); compare **in-sample AIC** within each granularity (do not rank AIC across hourly vs daily—different series and scales).

The Holt-Winters exponential smoothing model, often referred to as Triple Exponential Smoothing, is a popular time series forecasting method that extends simple and Holt's exponential smoothing to handle data with trend and seasonality. It's particularly useful for data where these components change over time.

Here's a breakdown of its key components:

- Level (L): This represents the average value of the series. It's smoothed using a parameter alpha (α).
- Trend (T): This captures the increasing or decreasing direction of the series. It's smoothed using a parameter beta (β).
- Seasonality (S): This accounts for repeating patterns in the data (e.g., hourly, daily, weekly, monthly cycles). It's smoothed using a parameter gamma (γ).

There are two main variations for combining the seasonal component:

1. Additive Seasonality (seasonal='add'): The seasonal component is added to the level and trend. This is suitable when the seasonal fluctuations are roughly constant over time.
- Example: If sales increase by a fixed amount every December, regardless of overall sales volume.
2. Multiplicative Seasonality (seasonal='mul'): The seasonal component is multiplied by the level and trend. This is appropriate when the seasonal fluctuations change proportionally with the level of the series.
- Example: If December sales are always 20% higher than the average, meaning the actual increase in units or dollars will be larger when overall sales are high.

In the provided notebook, the ExponentialSmoothing function from statsmodels implements this model, allowing you to specify the trend (here, 'add' for additive trend) and seasonal components, along with the seasonal_periods (e.g., 24 for hourly data with daily seasonality, 7 for daily data with weekly seasonality). The initialization_method='estimated' means the initial values for level, trend, and seasonality are estimated from the data, and optimized=True tells the model to find the best smoothing parameters (alpha, beta, gamma) that minimize the error.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from statsmodels.tsa.holtwinters import ExponentialSmoothing

DATA_PATH = Path("data/raw/de_lu_load_hourly.parquet")
if not DATA_PATH.is_file():
    DATA_PATH = Path(r"C:\Users\mhmta\Cursor_Study\energy-ts-fundamentals\data\raw\de_lu_load_hourly.parquet")

TZ = "Europe/Berlin"
H_FORECAST_HOURLY = 48

load_hourly = pd.read_parquet(DATA_PATH)
idx = load_hourly.index
if not isinstance(idx, pd.DatetimeIndex):
    load_hourly.index = pd.to_datetime(idx, utc=True)
if load_hourly.index.tz is None:
    load_hourly.index = load_hourly.index.tz_localize(
        TZ, ambiguous="infer", nonexistent="shift_forward"
    )
else:
    load_hourly.index = load_hourly.index.tz_convert(TZ)

y_hourly = (
    load_hourly["load_mw"]
    .sort_index()
    .asfreq("1h")
    .interpolate(limit_direction="both")
    .astype(float)
)

hourly_fit_endog = y_hourly.iloc[:-H_FORECAST_HOURLY]
hourly_holdout = y_hourly.iloc[-H_FORECAST_HOURLY:]
print("Hourly rows:", len(y_hourly), "| fit:", len(hourly_fit_endog), "| holdout:", len(hourly_holdout))

In [ ]:
def fit_hw(endog: pd.Series, seasonal_periods: int, seasonal: str):
    """Triple exponential smoothing: additive trend + chosen seasonal flavour."""
    model = ExponentialSmoothing(
        endog,
        trend="add",
        seasonal=seasonal,
        seasonal_periods=seasonal_periods,
        initialization_method="estimated",
    )
    return model.fit(optimized=True)


hourly_results = {}
for seas in ("add", "mul"):
    hourly_results[seas] = fit_hw(hourly_fit_endog, seasonal_periods=24, seasonal=seas)

aic_hourly = pd.DataFrame(
    [
        {"granularity": "hourly", "seasonal_periods": 24, "seasonality": seas, "AIC": fitted.aic}
        for seas, fitted in hourly_results.items()
    ]
).sort_values("AIC")
'''
This code snippet calculates and displays the Akaike Information Criterion (AIC) for the hourly 
Holt-Winters exponential smoothing models. It iterates through the results of models fitted 
with different seasonalities ('add' and 'mul'), extracts the AIC value from each, and then 
organizes this information into a pandas DataFrame. Finally, it sorts the DataFrame by AIC 
in ascending order, so the model with the lowest AIC (which is generally preferred) appears 
at the top.
'''

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")  # sets a pandas option to control the display format of floating-point numbers in DataFrames and Series
print("Hourly Holt–Winters AIC (lower is better; same training slice):")
print("Hourly Holt–Winters AIC (lower is better; same training slice):")
display(aic_hourly)

best_key = str(aic_hourly.iloc[0]["seasonality"])
hourly_fc = hourly_results[best_key].forecast(H_FORECAST_HOURLY) # generates a forecast. It takes the best-performing Holt-Winters model for hourly data (identified by best_key, which was determined based on the lowest AIC) and uses its forecast() method to predict future values for H_FORECAST_HOURLY steps ahead.
hourly_fc.index = hourly_holdout.index # assigns the index of the hourly_holdout DataFrame (which contains the actual values for the forecasted period) to the newly generated forecast (hourly_fc).
print(f"\n48-step forecast uses best AIC seasonal mode: seasonal='{best_key}'")

In [ ]:
show_h = min(168, len(hourly_fit_endog))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly_fit_endog.index[-show_h:], hourly_fit_endog.iloc[-show_h:], label="train (tail)") # This plots the last show_h hours of the training data. The .index[-show_h:] gets the time stamps for the x-axis, and .iloc[-show_h:] gets the corresponding load values for the y-axis. It's labeled "train (tail)" for the legend.      
ax.plot(hourly_holdout.index, hourly_holdout, label="actual holdout")
ax.plot(hourly_fc.index, hourly_fc.values, linestyle="--", label=f"forecast (seasonal={best_key})")
ax.legend()
ax.set_title("Hourly load: last week of train, 48h holdout vs ETS forecast")
ax.set_ylabel("MW")
fig.autofmt_xdate()
plt.tight_layout()

In [ ]:
y_daily = y_hourly.resample("1D").sum().dropna()

# Weekly seasonality: hold out one full period and forecast seven days ahead
H_FORECAST_DAILY = 7 # represents the number of days you intend to forecast into the future 
if len(y_daily) <= H_FORECAST_DAILY + 2 * 7: #  For a Holt-Winters model with a weekly seasonality (seasonal_periods=7), it's generally recommended to have at least two full seasonal cycles before the forecast period to accurately estimate the seasonal components.
    raise ValueError("Need enough daily history for seasonal_periods=7 plus holdout.")
daily_fit_endog = y_daily.iloc[:-H_FORECAST_DAILY]
daily_holdout = y_daily.iloc[-H_FORECAST_DAILY:]

daily_results = {}
for seas in ("add", "mul"):
    daily_results[seas] = fit_hw(daily_fit_endog, seasonal_periods=7, seasonal=seas)

aic_daily = pd.DataFrame(
    [
        {"granularity": "daily.sum", "seasonal_periods": 7, "seasonality": seas, "AIC": fitted.aic}
        for seas, fitted in daily_results.items()
    ]
).sort_values("AIC")

print("Daily-sum Holt–Winters AIC:")
display(aic_daily)

best_d = str(aic_daily.iloc[0]["seasonality"]) # ["seasonality"]: From that selected row, this extracts the value from the 'seasonality' column. This will be either 'add' or 'mul', indicating the type of seasonal component that performed best.
daily_fc = daily_results[best_d].forecast(H_FORECAST_DAILY)
daily_fc.index = daily_holdout.index
print(f"Daily horizon = {H_FORECAST_DAILY} day(s); best AIC seasonal='{best_d}'")

In [ ]:
show_d = min(56, len(daily_fit_endog))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily_fit_endog.index[-show_d:], daily_fit_endog.iloc[-show_d:], label="train (tail)")
ax.plot(daily_holdout.index, daily_holdout, label="actual holdout")
ax.plot(daily_fc.index, daily_fc.values, linestyle="--", label=f"forecast (seasonal={best_d})")
ax.legend()
ax.set_title("Daily summed load (MWh-ish): tail of train vs holdout vs ETS")
ax.set_ylabel("Sum MW in day")
fig.autofmt_xdate()
plt.tight_layout()

In [ ]:
aic_compare = pd.concat([aic_hourly, aic_daily], ignore_index=True)
print("AIC ranked within hourly and within daily (lower is better for nested models on the same endog).")
print("Do not compare raw AIC numbers between hourly and daily—they are different variables and sample sizes.")
display(aic_compare)